# Initialize Prefect secret block `env-vars`

See: https://pforge-exchange2.astrium.eads.net/confluence/pages/viewpage.action?pageId=495624025

In [ ]:
# Imports
import json
import os
from prefect.blocks.system import Secret
from resources import utils

In [ ]:
if not utils.cluster_mode:
    raise RuntimeError("This notebook should only be run in cluster mode!")

In [ ]:
block_name = "env-vars"
try:
    existing_values = (await Secret.load(block_name)).get()
    dump = f"\n{json.dumps(dict(sorted(existing_values.items())), indent=2)}"
except ValueError:
    existing_values = {}
    dump = "None"
print(f"Existing values for {block_name!r}: {dump}")

In [ ]:
# New values. You can copy/paste existing values.

new_values = {
    # Token that was used to setup the Dask clusters.
    # See: https://gateway.dask.org/authentication.html#using-jupyterhub-s-authentication
    "JUPYTERHUB_API_TOKEN": "***",

    # Needed to run the performance indicator prefect flow
    # The values for the following fields should be taken from rs-infra-core inventory,
    # file rs-infra-core/inventory/sample/host_vars/setup/apps.yml.
    # There is a section named rs_performance_indicator. The values for the fields
    # are set at the cluster deployment. These values should be also used here
    # Here is the aforementioned section:
    # rs_performance_indicator:
    #  database:
    #    host: postgresql-cluster-rw.database.svc.cluster.local
    #    name: performance
    #    password: test
    #    username: test
    #    secret: pi-database-password
    "POSTGRES_HOST": "postgresql-cluster-rw.database.svc.cluster.local",
    "POSTGRES_USER": "performance",
    "POSTGRES_PASSWORD": "***",
    "POSTGRES_PORT": "5432", # normally, 5432
    "POSTGRES_PI_DB": "performance",
    
    # S3 bucket name and subfolder.
    # NOTE: the "share-bucket" block will be created automatically from these
    # variables. So if you change these variables, please also remove the
    # "share-bucket" block and it will be recreated.
    "PREFECT_BUCKET_NAME": "rs-dev-cluster-temp",
    "PREFECT_BUCKET_FOLDER": "prefect-share",

    # Additional Prefect messages
    "PREFECT_DEBUG_MODE": "1",

    # Internal osam url
    "RSPY_HOST_OSAM": "http://rs-server-osam.processing.svc.cluster.local:8080",
}

# Jupyter env vars to pass to Prefect and Dask
for key in [
    "DASK_GATEWAY_ADDRESS",
    "DASK_GATEWAY_PUBLIC",
    "RSPY_PREFECT_URL",
    "RSPY_UAC_CHECK_URL",
    "RSPY_WEBSITE",
    "TEMPO_ENDPOINT",
]:
    if value := os.environ.get(key):
        new_values[key] = value

print(f"New values for {block_name!r}:\n{json.dumps(new_values, indent=2)}")

In [ ]:
# Overwrite values ?
diff = utils.compare_dict(existing_values, new_values)
if not diff:
    print("No differences with existing values.")
else:
    print(diff)
    answer = input(f"\nOverwrite these values in Prefect block {block_name!r} (Y/n)?")

    if answer.lower() == "y":
        await Secret(value=new_values).save("env-vars", overwrite=True)
        print(f"{block_name!r} overwritten.")
        existing_values = new_values